# Python OOP Deep Dive & Internals

This notebook covers advanced Object-Oriented Programming concepts in Python, focusing on:
1.  **Magic Methods (Dunder Methods)**
2.  **Object Creation (`__new__` vs `__init__`)**
3.  **Encapsulation & Name Mangling**
4.  **Tricky Interview Questions**
5.  **Lead Data Engineer Patterns**
6.  **Advanced Tricks (Descriptors, MRO, Attribute Access)**
7.  **Deep Dive: `__new__` vs `__init__` Q&A**

## 1. Magic Methods (Dunder Methods)

Magic methods allow objects to behave like built-in types (e.g., numbers, lists).

In [ ]:
import math

class Vector:
    """A simple 2D Vector demonstrating operator overloading."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"
    
    def __str__(self):
        return f"({self.x}, {self.y})"
    
    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)
    
    def __eq__(self, other):
        if not isinstance(other, Vector):
            return False
        return self.x == other.x and self.y == other.y

# Experiment here
v1 = Vector(2, 3)
v2 = Vector(3, 4)
print(f"v1 + v2 = {v1 + v2}")
print(f"v1 == v2: {v1 == v2}")

### Container Emulation
Making an object behave like a list using `__len__`, `__getitem__`, and `__call__`.

In [ ]:
class CustomList:
    def __init__(self, items):
        self._items = items
    
    def __len__(self):
        return len(self._items)
    
    def __getitem__(self, index):
        return self._items[index]
        
    def __call__(self):
        print(f"I hold {len(self)} items: {self._items}")

cl = CustomList([10, 20, 30])
print(f"Length: {len(cl)}")
print(f"Item at 1: {cl[1]}")
cl()  # Calling the object!

## 2. Object Creation (`__new__` vs `__init__`)

- `__new__`: Creates the instance (Allocator).
- `__init__`: Initializes the instance (Constructor).

In [ ]:
class Singleton:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("Creating new instance...")
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self, value):
        # WARNING: __init__ runs every time!
        self.value = value
        print(f"Initializing with {value}")

s1 = Singleton(10)
s2 = Singleton(20)

print(f"s1 is s2: {s1 is s2}")
print(f"s1.value: {s1.value}")  # What will this be?

## 3. Encapsulation & Name Mangling

Accessing "private" variables using the name mangling trick.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self._balance = balance       # Protected
        self.__secret_code = "1234"   # Private

a = BankAccount(100)

try:
    print(a.__secret_code)
except AttributeError as e:
    print(f"Error: {e}")

# The trick:
print(f"Mangled access: {a._BankAccount__secret_code}")

## 4. Tricky Questions

**Q: What happens if `__new__` returns an instance of a different class?**

In [ ]:
class A:
    def __new__(cls):
        return B()

class B:
    pass

obj = A()
print(f"Type of obj: {type(obj)}")
# Note: A.__init__ is NOT called.

## 5. Lead Data Engineer Patterns

### Mixins for Data Pipelines

In [ ]:
class LoggingMixin:
    def log(self, message):
        print(f"[LOG] {self.__class__.__name__}: {message}")

class ValidationMixin:
    def validate(self, data):
        if not data:
            raise ValueError("Data is empty")
        self.log("Data validated")

class ETLJob(LoggingMixin, ValidationMixin):
    def run(self, data):
        self.log("Starting job")
        self.validate(data)
        self.log("Job finished")

job = ETLJob()
job.run(["data"])

## 6. Advanced Tricks (Descriptors, MRO, Attribute Access)

### 6.1 `__getattr__` vs `__getattribute__`
- `__getattr__`: Called ONLY when attribute lookup FAILS.
- `__getattribute__`: Called for EVERY attribute access. **Danger of infinite recursion!**

In [ ]:
class Dangerous:
    def __getattribute__(self, name):
        print(f"Accessing {name}")
        # return self.name  # <--- INFINITE RECURSION!
        return super().__getattribute__(name)  # Correct way

d = Dangerous()
try:
    d.x
except AttributeError:
    print("AttributeError caught (expected)")

### 6.2 Descriptors
The mechanism behind `@property`, `classmethod`, and `staticmethod`.
An object that defines `__get__`, `__set__`, or `__delete__`.

In [ ]:
class IntegerField:
    """Descriptor that enforces integer type."""
    def __init__(self, name):
        self.name = name
    
    def __get__(self, instance, owner):
        return instance.__dict__.get(self.name)
    
    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} must be an integer")
        instance.__dict__[self.name] = value

class User:
    age = IntegerField("age")
    
    def __init__(self, age):
        self.age = age

u = User(30)
print(f"Age: {u.age}")
try:
    u.age = "thirty"  # Raises TypeError
except TypeError as e:
    print(f"Error: {e}")

### 6.3 MRO (Method Resolution Order)
How Python finds methods in multiple inheritance (The Diamond Problem).

In [ ]:
class A:
    def greet(self):
        print("A")

class B(A):
    def greet(self):
        print("B")
        super().greet()

class C(A):
    def greet(self):
        print("C")
        super().greet()

class D(B, C):
    def greet(self):
        print("D")
        super().greet()

print(f"MRO: {D.mro()}")
print("\nCalling D.greet():")
d = D()
d.greet()

## 7. Deep Dive: `__new__` vs `__init__` Q&A

**Q1: Can `__init__` return a value?**
- **Answer**: NO. It must return `None`. If you try to return something, it raises a `TypeError`.

In [ ]:
class BadInit:
    def __init__(self):
        return 10  # TypeError!

try:
    b = BadInit()
except TypeError as e:
    print(f"Caught expected error: {e}")

**Q2: Why do immutable types (int, str, tuple) use `__new__`?**
- **Answer**: Because `__init__` is too late! The object is already created by the time `__init__` runs. Since you can't modify an immutable object after creation, you must customize the *creation* step itself.

In [ ]:
class UpperString(str):
    def __new__(cls, value):
        # Modify the value BEFORE the object is created
        return super().__new__(cls, value.upper())

s = UpperString("hello")
print(f"UpperString: {s}")

**Q3: When is `__init__` NOT called?**
- **Answer**: If `__new__` returns an object that is NOT an instance of the class, `__init__` is skipped.

In [ ]:
class Skipper:
    def __new__(cls):
        print("__new__ called")
        return 123  # Returning an int, not a Skipper instance
    
    def __init__(self):
        print("__init__ called")  # This will NEVER print

obj = Skipper()
print(f"Object is: {obj} (Type: {type(obj)}) ")

**Q4: What are the arguments to `__new__` vs `__init__`?**
- `__new__` takes `cls` as the first argument (it's a static method).
- `__init__` takes `self` as the first argument (instance method).
- Both receive the same arguments passed to the class constructor.